# Pandora ZX validation

Correctness checks for the Pandora ZX simplification implementation. Requires Pandora container to be running, e.g. by `docker compose up`

In [ ]:
import io
import random
import time
import math
from contextlib import asynccontextmanager, contextmanager, nullcontext, redirect_stdout
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Iterator

import numpy as np
import pandas as pd
import pyzx as zx
import qiskit.qasm2
from IPython.display import display
from mqt.qcec import verify
from mqt.qcec.pyqcec import EquivalenceCriterion
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import SwapGate, QFT, DraperQFTAdder, CDKMRippleCarryAdder, VBERippleCarryAdder, HRSCumulativeMultiplier, RGQFTMultiplier, QuantumVolume, IQP, GroverOperator, qaoa_ansatz

from qiskit.quantum_info import Clifford, Operator, random_clifford

from pandora.db.core import PandoraDB
from pandora.db.repository import GateRepository
from pandora.db.service import PandoraService


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "pandora").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the Pandora project root")

PROJECT_ROOT = find_project_root()
CIRCUITS_ROOT = PROJECT_ROOT / "circuits"
OPTIMIZER_ROOT = CIRCUITS_ROOT / "optimizer"
CONFIG_PATH = PROJECT_ROOT / "default_config.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"Database config: {CONFIG_PATH}")

### Utilities

In [ ]:
SWAP_INSTRUCTION = qiskit.qasm2.CustomInstruction(
    "swap", 0, 2, SwapGate, builtin=True
)


def resolve_before_circuit(relative_path: str) -> Path:
    optimizer_root = OPTIMIZER_ROOT.resolve()
    path = (optimizer_root / relative_path).resolve()
    if optimizer_root not in path.parents:
        raise ValueError(f"Circuit must be inside {optimizer_root}")
    if not path.name.endswith("_before"):
        raise ValueError(f"Expected an exact *_before file, got {path.name!r}")
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


def load_pyzx_circuit(path: Path) -> zx.Circuit:
    return zx.Circuit.load(str(path)).to_basic_gates()


def pyzx_to_qiskit(circuit: zx.Circuit) -> QuantumCircuit:
    return qiskit.qasm2.loads(
        circuit.to_qasm(), custom_instructions=[SWAP_INSTRUCTION]
    )


def qiskit_to_pyzx(circuit: QuantumCircuit) -> zx.Circuit:
    return zx.Circuit.from_qasm(qiskit.qasm2.dumps(circuit)).to_basic_gates()


def graph_to_qiskit(graph) -> QuantumCircuit:
    return pyzx_to_qiskit(zx.extract_circuit(graph.copy()))


@asynccontextmanager
async def open_pandora_service():
    db = PandoraDB(str(CONFIG_PATH))
    await db.connect()
    try:
        yield PandoraService(db=db, repo=GateRepository(db))
    finally:
        await db.close()


def simplify_pyzx(graph, mode: str) -> None:
    if mode == "basic":
        zx.simplify.basic_simp(graph)
    elif mode == "full":
        zx.simplify.full_reduce(graph)
    else:
        raise ValueError(f"Unknown simplification mode: {mode}")


async def simplify_pandora(service: PandoraService, mode: str) -> None:
    if mode == "basic":
        await service.simplify_zx_basic()
    elif mode == "full":
        await service.simplify_zx_full()
    else:
        raise ValueError(f"Unknown simplification mode: {mode}")


### Cases and equivalence checks

In [ ]:
@dataclass
class Case:
    name: str
    circuit: QuantumCircuit
    pyzx_circuit: zx.Circuit
    truth_table: Clifford | None

    @property
    def is_clifford(self) -> bool:
        return self.truth_table is not None

    @property
    def qubits(self) -> int:
        return self.pyzx_circuit.qubits

    @property
    def gates(self) -> int:
        return len(self.pyzx_circuit.gates)


def stabiliser_truth_table(circuit: QuantumCircuit) -> Clifford | None:
    try:
        return Clifford(circuit)
    except Exception:
        return None


def extract_circuit_for_check(graph, reduce_first: bool = True) -> QuantumCircuit:
    target = graph.copy()
    if reduce_first:
        zx.simplify.full_reduce(target)
    return graph_to_qiskit(target)


def graph_stabiliser_truth_table(graph, reduce_first: bool = True) -> Clifford | None:
    return stabiliser_truth_table(extract_circuit_for_check(graph, reduce_first))


class TensorTooLarge(Exception):
    """A ZX tensor contraction would exceed TENSOR_MAX_RANK."""


_np_tensordot = np.tensordot


def _bounded_tensordot(a, b, axes=2):
    a, b = np.asarray(a), np.asarray(b)
    contracted = axes if isinstance(axes, int) else (
        1 if isinstance(axes[0], int) else len(axes[0])
    )
    rank = a.ndim + b.ndim - 2 * contracted
    if rank > TENSOR_MAX_RANK:
        raise TensorTooLarge(
            f"contraction needs rank {rank} ({2 ** rank * 16 / 2 ** 30:.1f} GiB), "
            f"TENSOR_MAX_RANK={TENSOR_MAX_RANK}"
        )
    return _np_tensordot(a, b, axes)


@contextmanager
def bounded_tensor_contraction():
    zx.tensor.np.tensordot = _bounded_tensordot
    try:
        yield
    finally:
        zx.tensor.np.tensordot = _np_tensordot


def compare_tensors_safe(expected, actual, description: str) -> bool | None:
    try:
        with bounded_tensor_contraction():
            equal = zx.compare_tensors(
                zx.tensorfy(expected, preserve_scalar=False),
                zx.tensorfy(actual, preserve_scalar=False),
            )
    except (MemoryError, ValueError, TensorTooLarge) as e:
        print(f"WARN: cannot compare tensors ({description}): {e}")
        return None
    if not equal:
        raise AssertionError(f"Tensors differ: {description}")
    return True


def assert_circuit_equivalent(reference, candidate) -> str:
    reference_tableau = stabiliser_truth_table(reference)   
    if reference_tableau is not None:
        if reference_tableau == stabiliser_truth_table(candidate):
            return "stabiliser_truth_table"
        raise AssertionError("Stabiliser truth tables differ")

    if reference.num_qubits <= UNITARY_MAX_QUBITS:
        if Operator(reference).equiv(Operator(candidate)):
            return "unitary"
        raise AssertionError("Unitaries differ")

    result = verify(reference.copy(), candidate.copy())
    if result.equivalence in {
        EquivalenceCriterion.equivalent,
        EquivalenceCriterion.equivalent_up_to_global_phase,
    }:
        return "qcec"
    raise AssertionError(f"Circuits are not equivalent (QCEC: {result})")


### Clifford circuit generators

In [ ]:
# Clifford gates Pandora can load (see QISKIT_TO_PANDORA in translator.py).
CLIFFORD_BASIS = ["rx", "ry", "rz", "h", "s", "sdg", "x", "y", "z", "cx", "cz", "swap",  "t", "tdg"]

SINGLE_QUBIT_CLIFFORDS = ["h", "s", "sdg", "x", "y", "z"]

def random_sequence_circuit(n: int, depth: int, rng: random.Random) -> QuantumCircuit:
    circuit = QuantumCircuit(n)
    for _ in range(depth):
        if n >= 2:
            gate = rng.choice(SINGLE_QUBIT_CLIFFORDS + ["cx", "cz", "swap"])
        else:
            gate = rng.choice(SINGLE_QUBIT_CLIFFORDS)
        if gate in SINGLE_QUBIT_CLIFFORDS:
            getattr(circuit, gate)(rng.randrange(n))
        else:
            a, b = rng.sample(range(n), 2)
            getattr(circuit, gate)(a, b)
    return circuit

def clean_qiskit_index(circuit: QuantumCircuit) -> QuantumCircuit:
    # Qiskit can produce qubits whose `_index is None`, which Pandora's translator dereferences.
    out = QuantumCircuit(circuit.num_qubits)
    for instruction in circuit.data:
        out.append(
            instruction.operation,
            [circuit.find_bit(q).index for q in instruction.qubits],
        )
    return out


def uniform_random_clifford_circuit(n: int, rng: random.Random) -> QuantumCircuit:
    circuit = random_clifford(n, seed=rng.randrange(2**31)).to_circuit()
    circuit = transpile(circuit, basis_gates=CLIFFORD_BASIS, optimization_level=0)
    return clean_qiskit_index(circuit)

def structured_clifford_circuits(qubit_counts: list[int]) -> Iterator[tuple[str, QuantumCircuit]]:
    for n in qubit_counts:  # GHZ preparation
        c = QuantumCircuit(n)
        c.h(0)
        for q in range(n - 1):
            c.cx(q, q + 1)
        yield f"ghz_{n}", c

    for n in qubit_counts:  # CNOT ladder back and forth (cancels to identity)
        c = QuantumCircuit(n)
        for q in range(n - 1):
            c.cx(q, q + 1)
        for q in reversed(range(n - 1)):
            c.cx(q, q + 1)
        yield f"cnot_ladder_{n}", c

    for n in qubit_counts:  # graph-state-like block of H / CZ / S layers
        c = QuantumCircuit(n)
        for q in range(n):
            c.h(q)
        for q in range(0, n - 1, 2):
            c.cz(q, q + 1)
        for q in range(n):
            c.s(q)
        for q in range(1, n - 1, 2):
            c.cz(q, q + 1)
        yield f"graph_state_{n}", c

    for n in qubit_counts:  # full SWAP network (reverses qubit order)
        c = QuantumCircuit(n)
        for i in range(n // 2):
            c.swap(i, n - 1 - i)
        yield f"swap_network_{n}", c


In [ ]:
def case_from_file(relative_path: str) -> Case:
    pyzx_circuit = load_pyzx_circuit(resolve_before_circuit(relative_path))
    circuit = pyzx_to_qiskit(pyzx_circuit)
    return Case(
        name=relative_path,
        circuit=circuit,
        pyzx_circuit=pyzx_circuit,
        truth_table=stabiliser_truth_table(circuit),
    )


def case_from_qiskit(name: str, circuit: QuantumCircuit) -> Case:
    return Case(
        name=name,
        circuit=clean_qiskit_index(circuit),
        pyzx_circuit=qiskit_to_pyzx(circuit),
        truth_table=stabiliser_truth_table(circuit),
    )


def build_clifford_cases(
    structure_circuit_qubits: list[int],
    uniform_widths: list[int],
    uniform_repeats: int,
    sequence_widths: list[int],
    sequence_repeats: int,
) -> list[Case]:
    rng = random.Random(123456)
    cases: list[Case] = []

    for n in uniform_widths:
        for r in range(uniform_repeats):
            cases.append(case_from_qiskit(
                f"clifford/uniform_n{n}_r{r}", uniform_random_clifford_circuit(n, rng)
            ))

    for n in sequence_widths:
        for r in range(sequence_repeats):
            cases.append(case_from_qiskit(
                f"clifford/seq_n{n}_d{n * n}_r{r}", random_sequence_circuit(n, n * n, rng)
            ))

    for name, circuit in structured_clifford_circuits(structure_circuit_qubits):
        cases.append(case_from_qiskit(f"clifford/{name}", circuit))

    return cases


def case_from_qiskit_generator(family: str, max_exponent :int, generator) -> list[Case]:
    cases: list[Case] = []
    for n in range(1, max_exponent + 1):
        i = 2**n
        circuit = generator(i)
        transpiled = transpile(circuit, basis_gates=CLIFFORD_BASIS, optimization_level=0)
        cases.append(case_from_qiskit(f"{family} {i}", transpiled))
    return cases


### Validations

In [ ]:
async def validate_circuit_to_zx(service: PandoraService, case: Case) -> dict:
    expected_graph = case.pyzx_circuit.to_graph()

    await service.build_circuit(case.circuit.copy())
    await service.convert_linked_circuit_to_zx()
    actual_graph = await service.export_to_pyzx()

    tensor_checked = compare_tensors_safe(
        expected_graph, actual_graph, f"circuit → ZX for {case.name}"
    )

    stt_checked = False
    if case.is_clifford:
        if graph_stabiliser_truth_table(actual_graph) != case.truth_table:
            raise AssertionError(f"Pandora circuit → ZX changed the truth table for {case.name}")
        stt_checked = True

    if tensor_checked is None and not stt_checked:
        raise AssertionError(f"No feasible equivalence check for {case.name}")

    return {
        "circuit": case.name,
        "qubits": case.qubits,
        "gates": case.gates,
        "clifford": case.is_clifford,
        "tensor_checked": bool(tensor_checked),
        "stt_checked": stt_checked,
    }


async def validate_simplification(
    service: PandoraService, case: Case, mode: str
) -> dict:
    input_graph = case.pyzx_circuit.to_graph()

    expected_graph = input_graph.copy()
    simplify_pyzx(expected_graph, mode)

    await service.build_pandora()
    await service.load_from_pyzx(input_graph.copy())
    await simplify_pandora(service, mode)
    actual_graph = await service.export_to_pyzx()

    method = assert_circuit_equivalent(
        case.circuit, graph_to_qiskit(actual_graph)
    )

    tensor_orig = compare_tensors_safe(
        input_graph, actual_graph, f"Pandora {mode} rewrite for {case.name}"
    )
    tensor_pyzx = compare_tensors_safe(
        expected_graph, actual_graph, f"Pandora {mode} vs PyZX {mode} for {case.name}"
    )

    return {
        "circuit": case.name,
        "mode": mode,
        "qubits": case.qubits,
        "gates": case.gates,
        "clifford": case.is_clifford,
        "equivalence_check": method,
        "tensor_checked_original": bool(tensor_orig),
        "tensor_checked_pyzx": bool(tensor_pyzx),
    }

### Validation runner

In [ ]:
async def run_cases(cases: list[Case], validate, quiet: bool = True) -> tuple[list[dict], list[dict]]:
    results: list[dict] = []
    failures: list[dict] = []

    async with open_pandora_service() as service:
        for index, case in enumerate(cases, start=1):
            label = (f"[{index}/{len(cases)}] {case.name} "
                     f"(qubits={case.qubits}, gates={case.gates})")
            buffer = io.StringIO()
            capture = redirect_stdout(buffer) if quiet else nullcontext()
            start = time.time()
            try:
                with capture:
                    row = await validate(service, case)
                row["seconds"] = round(time.time() - start, 3)
                results.append(row)
                print(f"{label}: OK ({row['seconds']:.2f}s)")
            except Exception as e:
                failures.append({"circuit": case.name, "error": repr(e)})
                print(f"{label}: FAILED: {e!r}")
            finally:
                warnings = [line for line in buffer.getvalue().splitlines()
                            if line.startswith("WARN:")]
                for line in warnings:
                    print(f"    {line}")

    print(f"\npassed: {len(results)}  failed: {len(failures)}")
    return results, failures

def show(results: list[dict] | None, failures: list[dict] | None, show_all: bool = False) -> None:
    if results and show_all:
        display(pd.DataFrame(results))
    if failures:
        print("Failures:")
        display(pd.DataFrame(failures))
    elif results:
        print("All pass!")


In [ ]:
def iqp(n: int) -> QuantumCircuit:
    # Dense, deterministic IQP instance.
    interactions = np.ones((n, n), dtype=int)
    return IQP(interactions)


def grover(n: int) -> QuantumCircuit:
    # Mark |11...1>.
    oracle = QuantumCircuit(n)
    oracle.h(n - 1)
    oracle.mcx(list(range(n - 1)), n - 1)
    oracle.h(n - 1)
    return GroverOperator(oracle)

In [ ]:
def _qaoa(edges, n):
    cost = SparsePauliOp.from_sparse_list(
        [("ZZ", [i, j], -0.5) for i, j in edges],
        num_qubits=n,
    )

    circuit = qaoa_ansatz(cost, reps=1, flatten=True)

    return circuit.assign_parameters({
        parameter: math.pi / 7
        for parameter in circuit.parameters
    })


def qaoa_cycle(n):
    edges = [
        (i, (i + 1) % n)
        for i in range(n)
    ]
    return _qaoa(edges, n)


def qaoa_random_sparse(n):
    rng = random.Random(42)

    edges = [
        (i, j)
        for i in range(n)
        for j in range(i + 1, n)
        if rng.random() < min(1.0, 4 / (n - 1))
    ]

    return _qaoa(edges, n)


def qaoa_complete(n):
    edges = [
        (i, j)
        for i in range(n)
        for j in range(i + 1, n)
    ]

    return _qaoa(edges, n)

### Validation run configuration

Non-Clifford circuits are checked by tensor comparison and QCEC, which are memory-intensive, so keep that list small. Clifford circuits are checked by stabiliser truth table and can be much larger.

Full coverage (the whole 11520-element 2-qubit group, 100/200-qubit random Cliffords) takes hours; the defaults below are the quick version.

In [ ]:
TENSOR_MAX_RANK = 29

# `Operator(circuit)` is a dense 2**n x 2**n matrix: 12 -> 0.25 GiB, 16 -> 64 GiB.
UNITARY_MAX_QUBITS = 12


FILE_CIRCUITS = [
    "Arithmetic_and_Toffoli/barenco_tof_3_before",
    "Arithmetic_and_Toffoli/gf2^E4_mult_before",
    "Arithmetic_and_Toffoli/mod5_4_before",
    "Arithmetic_and_Toffoli/tof_3_before",
    "Arithmetic_and_Toffoli/tof_4_before",
    "Arithmetic_and_Toffoli/vbe_adder_3_before",
    "QFT_and_Adders/QFT8_before",
    "Arithmetic_and_Toffoli/mod_mult_55_before",
]

TO_ZX_BASIC_CASES = [case_from_file(path) for path in FILE_CIRCUITS] \
    + case_from_qiskit_generator("QFT", 3, QFT) \
    + case_from_qiskit_generator("Draper QFT Adder", 2, DraperQFTAdder) \
    + case_from_qiskit_generator("CDKM Adder", 2, CDKMRippleCarryAdder) \
    + case_from_qiskit_generator("VBE Adder", 1, VBERippleCarryAdder) \
    + case_from_qiskit_generator("HRS Cumulative Multiplier", 1, HRSCumulativeMultiplier) \
    + case_from_qiskit_generator("RGQFT Multiplier", 1, RGQFTMultiplier) \
    + case_from_qiskit_generator("QAOA Cycle", 3, qaoa_cycle) \
    + case_from_qiskit_generator("QAOA Random Sparse", 3, qaoa_random_sparse) \
    + case_from_qiskit_generator("QAOA Complete", 3, qaoa_complete) \
    + case_from_qiskit_generator("Quantum Volume", 3, QuantumVolume) \
    + case_from_qiskit_generator("IQP", 2, iqp) \
    + case_from_qiskit_generator("Grover Operator", 2, grover)

SIMPLIFY_BASIC_CASES = [case_from_file(path) for path in FILE_CIRCUITS] \
    + case_from_qiskit_generator("QFT", 2, QFT) \
    + case_from_qiskit_generator("Draper QFT Adder", 1, DraperQFTAdder) \
    + case_from_qiskit_generator("CDKM Adder", 1, CDKMRippleCarryAdder) \
    + case_from_qiskit_generator("VBE Adder", 1, VBERippleCarryAdder) \
    + case_from_qiskit_generator("HRS Cumulative Multiplier", 1, HRSCumulativeMultiplier) \
    + case_from_qiskit_generator("RGQFT Multiplier", 1, RGQFTMultiplier) \
    + case_from_qiskit_generator("QAOA Cycle", 3, qaoa_cycle) \
    + case_from_qiskit_generator("QAOA Random Sparse", 3, qaoa_random_sparse) \
    + case_from_qiskit_generator("QAOA Complete", 3, qaoa_complete) \
    + case_from_qiskit_generator("Quantum Volume", 2, QuantumVolume) \
    + case_from_qiskit_generator("IQP", 2, iqp) \
    + case_from_qiskit_generator("Grover Operator", 2, grover)

TO_ZX_CLIFFORD_CASES = build_clifford_cases(
    structure_circuit_qubits=[2, 8, 16, 64, 3, 9, 21, 65],
    uniform_widths=[2, 8, 16, 3, 9, 21],
    uniform_repeats=3,
    sequence_widths=[2, 8, 16, 3, 9, 21],
    sequence_repeats=3
)

SIMPLIFY_CLIFFORD_CASES = build_clifford_cases(
    structure_circuit_qubits=[2, 8, 16, 3, 9, 21],
    uniform_widths=[2, 8, 3, 9],
    uniform_repeats=3,
    sequence_widths=[2, 8, 3, 9],
    sequence_repeats=3
)


## 1. Pandora circuit to ZX validation

In [ ]:
conversion_results, conversion_failures = await run_cases(TO_ZX_BASIC_CASES, validate_circuit_to_zx)

In [ ]:
clifford_conversion_results, clifford_conversion_failures = await run_cases(TO_ZX_CLIFFORD_CASES, validate_circuit_to_zx)

## 2. Pandora basic and full rewrite validation

In [ ]:
rewrite_results = []
rewrite_failures = []

for mode in ("basic", "full"):
    print(f"=== {mode} ===")
    results, failures = await run_cases(
        SIMPLIFY_BASIC_CASES,
        partial(validate_simplification, mode=mode),
    )
    rewrite_results += results
    rewrite_failures += failures

In [ ]:
clifford_rewrite_results = []
clifford_rewrite_failures = []

for mode in ("basic", "full"):
    print(f"=== {mode} ===")
    results, failures = await run_cases(
        SIMPLIFY_CLIFFORD_CASES, partial(validate_simplification, mode=mode)
    )
    clifford_rewrite_results += results
    clifford_rewrite_failures += failures

## Result summary

In [ ]:
show(conversion_results, conversion_failures)
show(clifford_conversion_results, clifford_conversion_failures)
show(rewrite_results, rewrite_failures)
show(clifford_rewrite_results, clifford_rewrite_failures)